Retrieve data

In [1]:
import requests
import pandas as pd
import numpy as np
import holidays

# Define date ranges
train_start_date = '2015-01-01'
train_end_date = '2023-12-31'
test_start_date = '2024-01-01'
test_end_date = '2024-12-31'

# Fetch data from the API for training and testing
url_train = f"https://api.energy-charts.info/price?bzn=DE-LU&start={train_start_date}&end={train_end_date}"
url_test = f"https://api.energy-charts.info/price?bzn=DE-LU&start={test_start_date}&end={test_end_date}"

response_train = requests.get(url_train)
response_test = requests.get(url_test)

data_train = response_train.json()
data_test = response_test.json()

# Convert timestamps to UTC, then to Germany time (Europe/Berlin)
train_timestamps = pd.to_datetime(data_train['unix_seconds'], unit='s', utc=True).tz_convert('Europe/Berlin')
test_timestamps = pd.to_datetime(data_test['unix_seconds'], unit='s', utc=True).tz_convert('Europe/Berlin')

# Convert to DataFrame
train_df = pd.DataFrame({
    'timestamp': train_timestamps,
    'price': data_train['price']
})
test_df = pd.DataFrame({
    'timestamp': test_timestamps,
    'price': data_test['price']
})

# Remove timezone information while keeping local time
train_df['timestamp'] = train_df['timestamp'].dt.tz_localize(None)
test_df['timestamp'] = test_df['timestamp'].dt.tz_localize(None)

# Clean the data (remove rows with null prices)
train_df = train_df.dropna()
test_df = test_df.dropna()

# Rename columns for NeuralForecast
train_df = train_df.rename(columns={'timestamp': 'ds', 'price': 'y'})
test_df = test_df.rename(columns={'timestamp': 'ds', 'price': 'y'})

# Add unique_id column
train_df['unique_id'] = 'electricity_prices'
test_df['unique_id'] = 'electricity_prices'


In [2]:
train_df.head()

,ds,y,unique_id
32854,2018-10-01 00:00:00,59.53,electricity_prices
32855,2018-10-01 01:00:00,56.10,electricity_prices
32856,2018-10-01 02:00:00,51.41,electricity_prices
32857,2018-10-01 03:00:00,47.38,electricity_prices
32858,2018-10-01 04:00:00,47.59,electricity_prices


In [3]:
test_df.head()

,ds,y,unique_id
0,2024-01-01 00:00:00,0.10,electricity_prices
1,2024-01-01 01:00:00,0.01,electricity_prices
2,2024-01-01 02:00:00,0.00,electricity_prices
3,2024-01-01 03:00:00,-0.01,electricity_prices
4,2024-01-01 04:00:00,-0.03,electricity_prices


Add calendar features

In [4]:

# Define the cyclic encoding function
def cyclicEncode(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col] / max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col] / max_val)
    return data

# Add calendar features
def add_calendar_features(df):
    german_holidays = holidays.Germany(years=df['ds'].dt.year.unique())
    
    # Basic date features
    df['day_of_week'] = df['ds'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['month'] = df['ds'].dt.month
    df['hour'] = df['ds'].dt.hour
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Holiday feature
    df['is_holiday'] = df['ds'].apply(lambda x: int(x in german_holidays))
    
    # Add cyclic encoding for month, day_of_week, and hour
    df = cyclicEncode(df, 'month', 12)
    df = cyclicEncode(df, 'day_of_week', 7)
    df = cyclicEncode(df, 'hour', 24)
    
    return df


In [5]:

# Add calendar features to train and test datasets
train_df = add_calendar_features(train_df)
test_df = add_calendar_features(test_df)

# Save final processed DataFrames to CSV
train_df.to_csv('training_data_germany_time_zone_with_calendar.csv', index=False)
test_df.to_csv('testing_data_germany_time_zone_with_calendar.csv', index=False)

# Check date ranges
print("First date in train_df:", train_df['ds'].min())
print("Last date in train_df:", train_df['ds'].max())
print("First date in test_df:", test_df['ds'].min())
print("Last date in test_df:", test_df['ds'].max())


First date in train_df: 2018-10-01 00:00:00
Last date in train_df: 2023-12-31 23:00:00
First date in test_df: 2024-01-01 00:00:00
Last date in test_df: 2024-12-31 23:00:00
Train and test data processed and saved successfully.


In [6]:
train_df.head()

,ds,y,unique_id,day_of_week,month,hour,is_weekend,is_holiday,month_sin,month_cos,day_of_week_sin,day_of_week_cos,hour_sin,hour_cos
32854,2018-10-01 00:00:00,59.53,electricity_prices,0,10,0,0,0,-0.866025,0.5,0.0,1.0,0.000000,1.000000
32855,2018-10-01 01:00:00,56.10,electricity_prices,0,10,1,0,0,-0.866025,0.5,0.0,1.0,0.258819,0.965926
32856,2018-10-01 02:00:00,51.41,electricity_prices,0,10,2,0,0,-0.866025,0.5,0.0,1.0,0.500000,0.866025
32857,2018-10-01 03:00:00,47.38,electricity_prices,0,10,3,0,0,-0.866025,0.5,0.0,1.0,0.707107,0.707107
32858,2018-10-01 04:00:00,47.59,electricity_prices,0,10,4,0,0,-0.866025,0.5,0.0,1.0,0.866025,0.500000


In [7]:
test_df.head()

,ds,y,unique_id,day_of_week,month,hour,is_weekend,is_holiday,month_sin,month_cos,day_of_week_sin,day_of_week_cos,hour_sin,hour_cos
0,2024-01-01 00:00:00,0.10,electricity_prices,0,1,0,0,1,0.5,0.866025,0.0,1.0,0.000000,1.000000
1,2024-01-01 01:00:00,0.01,electricity_prices,0,1,1,0,1,0.5,0.866025,0.0,1.0,0.258819,0.965926
2,2024-01-01 02:00:00,0.00,electricity_prices,0,1,2,0,1,0.5,0.866025,0.0,1.0,0.500000,0.866025
3,2024-01-01 03:00:00,-0.01,electricity_prices,0,1,3,0,1,0.5,0.866025,0.0,1.0,0.707107,0.707107
4,2024-01-01 04:00:00,-0.03,electricity_prices,0,1,4,0,1,0.5,0.866025,0.0,1.0,0.866025,0.500000
